# 🧱 Lab: The Clean-Machine Release Audit

**Module 11: Ship It: Capstone, Docs, Responsible AI** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Run** a stranger-style audit: documented commands only, on a tree that is not your laptop's snowflake
2. **Verify** tests, CLI help, secret scan, licence note, and no absolute paths
3. **Keep** a PASS or FAIL log as the release artifact
4. **Fix** one planted failure and rerun the same check

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. | mini-repo-reproducibility-audit |
| Secret Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. | mini-init-git-repo |
| Unit Test | A small assertion about one behavior. A unit test that cannot fail is theatre, it should catch a real bug when the behavior breaks. | mini-first-pytest-tests |
| Black-Box Verification | Use only documented commands and interfaces to run the system like a new user, every stumble is a documentation bug or missing contract. | lab-black-box-run |
| Data Privacy and Licensing | Handling PII with consent and retention limits, while respecting dataset licenses, redistribution rights, and allowed uses across collection, training, and deployment. | lab-limitations-and-responsible-ai |
| Command-Line Interface | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. | mini-build-cli |

## Prerequisites

- **lab-black-box-run**: Use only documented commands; every stumble is a doc bug
- **lab-limitations-and-responsible-ai**: LIMITATIONS and the privacy-licence-secret checklist
- **mini-build-cli**: `--help`, exit codes, stdout vs stderr
- **proj-triage-capstone**: The repo shape you are auditing


## Why a clean machine

"Works on my machine" is not a release. A clean-machine audit asks: if someone clones this repo with nothing pre-installed, and they only follow the README, do tests and the CLI work? We will build two fixture trees — one broken, one shippable — and run the same checks on both.


In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

from pathlib import Path
import re
import subprocess
import sys
import tempfile


## Part 1: Two fixture repos on disk

The broken tree plants an absolute path, a key-shaped secret, a CLI with no `--help`, and no tests. The clean tree uses pathlib, `.env.example`, pytest, and a documented CLI. Both are support-triage shaped, not a different product.


In [2]:
def write_tree(root: Path, files: dict[str, str]) -> None:
    for rel, content in files.items():
        path = root / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")


BROKEN = {
    "README.md": "# Triage\n\nRun it.\n",
    "src/triage/__init__.py": "",
    "src/triage/cli.py": (
        "from pathlib import Path\n"
        "DATA = Path(r'C:\\Users\\alice\\support_inbox.csv')\n"
        "SECRET = 'sk-live-not-a-real-key-but-scans'\n"
        "if __name__ == '__main__':\n"
        "    print('no help')\n"
    ),
    ".env": "OPENAI_API_KEY=sk-live-not-a-real-key-but-scans\n",
    ".gitignore": "*.pyc\n",
}

CLI_PY = "\n".join([
    "import argparse",
    "from pathlib import Path",
    "import sys",
    "",
    "def main(argv=None):",
    "    parser = argparse.ArgumentParser(description='Triage support messages')",
    "    parser.add_argument('--input', type=Path, default=Path('data/sample.csv'))",
    "    args = parser.parse_args(argv)",
    "    if not args.input.exists():",
    "        print('missing ' + str(args.input), file=sys.stderr)",
    "        return 2",
    "    print('label,ticket_id')",
    "    print('Billing,1')",
    "    return 0",
    "",
    "if __name__ == '__main__':",
    "    raise SystemExit(main())",
    "",
])

TEST_PY = "\n".join([
    "from triage.cli import main",
    "",
    "def test_help_does_not_crash():",
    "    try:",
    "        main(['--help'])",
    "    except SystemExit as exc:",
    "        assert exc.code == 0",
    "",
    "def test_sample_csv_runs(tmp_path):",
    "    sample = tmp_path / 'one.csv'",
    "    sample.write_text('ticket_id,text\\n1,hi\\n', encoding='utf-8')",
    "    assert main(['--input', str(sample)]) == 0",
    "",
])

README_CLEAN = "\n".join([
    "# Support-message triage",
    "",
    "Label support tickets.",
    "",
    "## Can I run it?",
    "",
    "```bash",
    "python -m venv .venv",
    "pip install -r requirements.txt",
    "python -m pytest",
    "python -m triage.cli --help",
    "python -m triage.cli --input data/sample.csv",
    "```",
    "",
])

CLEAN = {
    "README.md": README_CLEAN,
    "LIMITATIONS.md": (
        "This is a demonstration project, not a production system.\n"
        "Dataset: course sample CSV, teaching use only. No PII.\n"
    ),
    "LICENSE": "MIT License\n",
    "requirements.txt": "pytest>=8\n",
    ".gitignore": ".env\n.venv\n__pycache__/\n",
    ".env.example": "OPENAI_API_KEY=your-key-here\n",
    "src/triage/__init__.py": "",
    "src/triage/cli.py": CLI_PY,
    "data/sample.csv": "ticket_id,text\n1,Please refund last month\n",
    "tests/test_cli.py": TEST_PY,
}

tmp = Path(tempfile.mkdtemp(prefix="triage-audit-"))
broken_root = tmp / "broken"
clean_root = tmp / "clean"
write_tree(broken_root, BROKEN)
write_tree(clean_root, CLEAN)
md(
    "Fixture roots\n\n"
    f"- broken: `{broken_root.as_posix()}`\n"
    f"- clean: `{clean_root.as_posix()}`"
)


Fixture roots

- broken: `C:/Users/molak/AppData/Local/Temp/triage-audit-d_p44odz/broken`
- clean: `C:/Users/molak/AppData/Local/Temp/triage-audit-d_p44odz/clean`

## Part 2: The audit checks

Black-box rule: a stranger only gets what the README documents. Each check returns `"PASS"` or `"FAIL"` with a reason. Absolute-path scan looks for `C:\` and `/Users/` in source files, not in this notebook's own strings about the broken fixture.


In [3]:
KEY_RE = re.compile(r"sk-[A-Za-z0-9]{8,}")
ABS_RE = re.compile(r"(C:\\Users\\|/Users/[A-Za-z])")
DOC_COMMANDS = ("pytest", "--help", "venv", "pip install")


def iter_text_files(root: Path):
    skip = {".venv", "__pycache__"}
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if any(part in skip for part in path.parts):
            continue
        if path.suffix.lower() in {".png", ".pyc"}:
            continue
        yield path


def check_readme_commands(root: Path) -> tuple[str, str]:
    readme = root / "README.md"
    if not readme.exists():
        return "FAIL", "no README.md"
    text = readme.read_text(encoding="utf-8").lower()
    missing = [c for c in DOC_COMMANDS if c not in text]
    if missing:
        return "FAIL", "README missing documented commands: " + ", ".join(missing)
    return "PASS", "clone-style commands are documented"


def check_secrets(root: Path) -> tuple[str, str]:
    hits = []
    gitignore = (root / ".gitignore").read_text(encoding="utf-8") if (root / ".gitignore").exists() else ""
    if ".env" not in gitignore:
        hits.append(".env not gitignored")
    if (root / ".env").exists():
        hits.append(".env file is in the tree")
    for path in iter_text_files(root):
        if path.name == ".env":
            continue
        text = path.read_text(encoding="utf-8", errors="replace")
        if KEY_RE.search(text) and path.name != ".env.example":
            hits.append(f"key-shaped secret in {path.relative_to(root).as_posix()}")
    return ("FAIL", "; ".join(hits)) if hits else ("PASS", "no key-shaped secrets; .env ignored")


def check_absolute_paths(root: Path) -> tuple[str, str]:
    hits = []
    for path in iter_text_files(root):
        if path.suffix not in {".py", ".md", ".txt", ".toml"}:
            continue
        text = path.read_text(encoding="utf-8", errors="replace")
        if ABS_RE.search(text):
            hits.append(path.relative_to(root).as_posix())
    return ("FAIL", "absolute paths in " + ", ".join(hits)) if hits else ("PASS", "no machine-local absolute paths")


def check_licence_privacy(root: Path) -> tuple[str, str]:
    blob = ""
    for name in ("LIMITATIONS.md", "README.md", "LICENSE"):
        p = root / name
        if p.exists():
            blob += p.read_text(encoding="utf-8")
    if "MIT" not in blob and "licence" not in blob.lower() and "license" not in blob.lower():
        return "FAIL", "no licence note"
    if "PII" not in blob and "demonstration" not in blob.lower():
        return "FAIL", "no privacy or demonstration note"
    return "PASS", "licence and demonstration or PII note present"


def check_tests(root: Path) -> tuple[str, str]:
    tests = list(root.glob("tests/test_*.py"))
    if not tests:
        return "FAIL", "no tests/test_*.py"
    env = {**dict(**__import__("os").environ), "PYTHONPATH": str(root / "src")}
    proc = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=root,
        env=env,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        return "FAIL", "pytest failed: " + (proc.stdout + proc.stderr)[-300:]
    return "PASS", "pytest exited 0"


def check_cli_help(root: Path) -> tuple[str, str]:
    env = {**dict(**__import__("os").environ), "PYTHONPATH": str(root / "src")}
    proc = subprocess.run(
        [sys.executable, "-m", "triage.cli", "--help"],
        cwd=root,
        env=env,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        return "FAIL", "CLI --help failed: " + (proc.stdout + proc.stderr)[-200:]
    if "usage" not in proc.stdout.lower() and "usage" not in proc.stderr.lower():
        return "FAIL", "CLI --help produced no usage text"
    return "PASS", "CLI --help exited 0"


def run_audit(root: Path) -> list[dict]:
    checks = [
        ("documented_commands", check_readme_commands),
        ("secret_scan", check_secrets),
        ("absolute_paths", check_absolute_paths),
        ("licence_privacy", check_licence_privacy),
        ("unit_tests", check_tests),
        ("cli_help", check_cli_help),
    ]
    rows = []
    for name, fn in checks:
        status, reason = fn(root)
        rows.append({"check": name, "status": status, "reason": reason})
    return rows


def render(title: str, rows: list[dict]) -> str:
    lines = [f"### {title}", "", "| Check | Status | Reason |", "|---|---|---|"]
    for row in rows:
        lines.append(f"| {row['check']} | **{row['status']}** | {row['reason']} |")
    n_fail = sum(1 for r in rows if r["status"] == "FAIL")
    lines.append("")
    lines.append(f"Summary: {len(rows) - n_fail} PASS, {n_fail} FAIL")
    return "\n".join(lines)


broken_rows = run_audit(broken_root)
clean_rows = run_audit(clean_root)
md(render("Broken fixture", broken_rows) + "\n\n" + render("Clean fixture", clean_rows))


### Broken fixture

| Check | Status | Reason |
|---|---|---|
| documented_commands | **FAIL** | README missing documented commands: pytest, --help, venv, pip install |
| secret_scan | **FAIL** | .env not gitignored; .env file is in the tree |
| absolute_paths | **FAIL** | absolute paths in src/triage/cli.py |
| licence_privacy | **FAIL** | no licence note |
| unit_tests | **FAIL** | no tests/test_*.py |
| cli_help | **FAIL** | CLI --help produced no usage text |

Summary: 0 PASS, 6 FAIL

### Clean fixture

| Check | Status | Reason |
|---|---|---|
| documented_commands | **PASS** | clone-style commands are documented |
| secret_scan | **PASS** | no key-shaped secrets; .env ignored |
| absolute_paths | **PASS** | no machine-local absolute paths |
| licence_privacy | **PASS** | licence and demonstration or PII note present |
| unit_tests | **PASS** | pytest exited 0 |
| cli_help | **PASS** | CLI --help exited 0 |

Summary: 6 PASS, 0 FAIL

## Part 3: Write the PASS or FAIL log

This is the release artifact. Copy the same script onto your capstone repo after the fixtures make sense.


In [4]:
log_lines = ["# Release audit log", "", "Target: clean fixture (stand-in for the capstone clone)", ""]
for row in clean_rows:
    log_lines.append(f"- {row['status']}: {row['check']} — {row['reason']}")
n_fail = sum(1 for r in clean_rows if r["status"] == "FAIL")
log_lines.append("")
log_lines.append("OVERALL: " + ("FAIL" if n_fail else "PASS"))

out_dir = Path("artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
log_path = out_dir / "RELEASE_AUDIT.md"
log_path.write_text("\n".join(log_lines) + "\n", encoding="utf-8")
md(f"Wrote `{log_path.as_posix()}`\n\n" + "\n".join(log_lines))


Wrote `artifacts/RELEASE_AUDIT.md`

# Release audit log

Target: clean fixture (stand-in for the capstone clone)

- PASS: documented_commands — clone-style commands are documented
- PASS: secret_scan — no key-shaped secrets; .env ignored
- PASS: absolute_paths — no machine-local absolute paths
- PASS: licence_privacy — licence and demonstration or PII note present
- PASS: unit_tests — pytest exited 0
- PASS: cli_help — CLI --help exited 0

OVERALL: PASS

## 🎯 Summary

### What You Built

You ran a six-check clean-machine audit on two support-triage fixture repos: documented commands, secret scan, absolute paths, licence and privacy notes, pytest, and CLI `--help`. The broken tree fails on purpose. The clean tree writes `artifacts/RELEASE_AUDIT.md` as the PASS or FAIL log.

### Key Takeaways

1. **Reproducibility** — a stranger gets the README, not your shell history
2. **Black-box verification** — only documented commands count; missing `--help` is a doc bug
3. **Secret hygiene** — `.env` in the tree or a key-shaped string is a FAIL
4. **Unit test plus CLI** — pytest and `--help` are the shippable entry points
5. **Data privacy and licensing** — a release log without a demonstration or PII note is incomplete
6. **Absolute paths** — `C:\Users\...` or `/Users/...` in source is a machine-local landmine
